# Lakebase 101 — Post-Deploy

Run this **after** `databricks bundle deploy`.

1. Grants the app's service principal `CAN_RUN` on the synced-table pipelines (the **Sync Now** button)
2. Grants the app SP **Unity Catalog** read access — the speed test & analytics panes query the gold/source tables via the SQL Warehouse *as the SP*
3. Grants **Postgres** schema access and creates + seeds the OLTP tables (`orders`, `inventory`) the app reads and writes


In [0]:
CATALOG = "lakebase_101_catalog"
APP_NAME = "lakebase-101-app"

In [0]:
"""Grant the app's service principal CAN_MANAGE_RUN on synced-table pipelines.
This allows the 'Sync Now' button in the app to trigger on-demand refreshes."""
import time
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.pipelines import PipelineAccessControlRequest, PipelinePermissionLevel

APP_NAME = "lakebase-101-app"

w = WorkspaceClient()

# Wait briefly for the app to be registered after deploy
for attempt in range(5):
    try:
        app = w.apps.get(APP_NAME)
        sp_name = app.service_principal_client_id
        print(f"App SP (client_id): {sp_name}")
        print(f"App SP (display):   {app.service_principal_name}")
        break
    except Exception:
        if attempt < 4:
            time.sleep(5)
        else:
            raise RuntimeError(f"App '{APP_NAME}' not found after deploy.")

# Find synced-table pipelines and grant permissions
granted = 0
for p in w.pipelines.list_pipelines(filter=f"name LIKE '%{CATALOG}%'"):
    try:
        w.pipelines.update_permissions(
            pipeline_id=p.pipeline_id,
            access_control_list=[
                PipelineAccessControlRequest(
                    service_principal_name=sp_name,
                    permission_level=PipelinePermissionLevel.CAN_RUN
                )
            ]
        )
        granted += 1
        print(f"  ✅ {p.pipeline_id} | {p.name}")
    except Exception as e:
        print(f"  ⚠️  {p.pipeline_id}: {e}")

print(f"\n✅ Granted CAN_MANAGE_RUN on {granted} pipeline(s) to {sp_name}")

App SP (client_id): cd61e919-c5b9-4b85-b2d1-c0887c411365
App SP (display):   app-1s0sa2 lakebase-101-app
  ✅ 37e0efa1-28b1-40cb-a4e3-29b7a176537d | Synced table: lakebase_101_catalog.lakebase_101_schema.sales_events_synced mYjtWR
  ✅ 6d14ef52-0c7c-4b1d-8478-dbf8571a8d69 | Synced table: lakebase_101_catalog.lakebase_101_schema.customers_directory_synced ogtv8l
  ✅ ac3b0830-391b-4774-9e57-89312424bfbf | Synced table: lakebase_101_catalog.lakebase_101_schema.customer_360_synced BdYv18

✅ Granted CAN_MANAGE_RUN on 3 pipeline(s) to cd61e919-c5b9-4b85-b2d1-c0887c411365


In [0]:
"""Grant the app's service principal Unity Catalog read access (via SQL).
The speed test (/api/speed) and analytics showdown (/api/aggregate) query the gold
and source tables through the SQL Warehouse *as the app SP*. Without USE CATALOG /
USE SCHEMA / SELECT those warehouse queries fail with a permission error.

Uses spark.sql GRANTs (not w.grants.update) to avoid SDK enum-serialization issues."""
SCHEMA = "lakebase_101_schema"
role = w.apps.get(APP_NAME).service_principal_client_id  # SP client_id = the UC principal

# Backtick-quote the principal (it's a UUID). SELECT on a schema covers all current
# and future tables in it.
spark.sql(f"GRANT USE CATALOG ON CATALOG {CATALOG} TO `{role}`")
spark.sql(f"GRANT USE SCHEMA  ON SCHEMA  {CATALOG}.{SCHEMA} TO `{role}`")
spark.sql(f"GRANT SELECT      ON SCHEMA  {CATALOG}.{SCHEMA} TO `{role}`")
print(f"✅ UC grants (USE CATALOG / USE SCHEMA / SELECT) applied to {role}")

✅ UC grants (USE CATALOG / USE SCHEMA / SELECT) applied to cd61e919-c5b9-4b85-b2d1-c0887c411365


In [0]:
"""Postgres setup for the app SP:
  1. USAGE + SELECT on the synced-table schema (so the app reads the reverse-ETL tables)
  2. Create the OLTP tables the app owns (orders, inventory) + grant the SP read/write
  3. Seed inventory from the products Delta table
Derives the Postgres role from the app's service_principal_client_id (no hardcoded UUIDs)."""
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade", "databricks-sdk>=0.118.0"])
# Evict old SDK modules so re-import picks up the upgraded version
for _k in list(sys.modules.keys()):
    if _k.startswith("databricks"):
        del sys.modules[_k]
from databricks.sdk import WorkspaceClient
w = WorkspaceClient()

import psycopg2

PROJECT_ID  = "lakebase-101-demo"
BRANCH_ID   = "production"
ENDPOINT_ID = "primary"
DATABASE    = "lakebase_101_db"
SCHEMA      = "lakebase_101_schema"

role = w.apps.get(APP_NAME).service_principal_client_id
print(f"App SP role (client_id): {role}")

# Endpoint host + a Lakebase-scoped JWT (we connect as the deploying user, who owns the schema)
endpoint_name = f"projects/{PROJECT_ID}/branches/{BRANCH_ID}/endpoints/{ENDPOINT_ID}"
ep = w.postgres.get_endpoint(name=endpoint_name)
host = ep.status.hosts.host
print(f"Lakebase host: {host}")

token = w.postgres.generate_database_credential(endpoint=endpoint_name).token
user = w.current_user.me().user_name

conn = psycopg2.connect(
    host=host, port=5432, dbname=DATABASE,
    user=user, password=token, sslmode="require", connect_timeout=10,
)
conn.autocommit = True

with conn.cursor() as cur:
    # 1. Read access to the synced-table schema (persists across syncs; DEFAULT PRIVILEGES
    #    covers tables a SNAPSHOT sync re-creates)
    cur.execute(f'GRANT USAGE ON SCHEMA {SCHEMA} TO "{role}";')
    cur.execute(f'GRANT SELECT ON ALL TABLES IN SCHEMA {SCHEMA} TO "{role}";')
    cur.execute(f'ALTER DEFAULT PRIVILEGES IN SCHEMA {SCHEMA} GRANT SELECT ON TABLES TO "{role}";')
    print(f"✅ Synced-table grants on schema {SCHEMA}")

    # 2. OLTP tables the app owns (place-order writes; stats tiles + customer-360 read).
    #    In public because the app queries them unqualified (search_path = "$user", public).
    #    GENERATED ALWAYS AS IDENTITY -> no sequence USAGE grant needed for the SP.
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.inventory (
          product_id    int PRIMARY KEY,
          product_name  text NOT NULL,
          category      text,
          price         numeric(10,2) NOT NULL,
          stock_on_hand int NOT NULL DEFAULT 0,
          updated_at    timestamptz NOT NULL DEFAULT now()
        );""")
    cur.execute("""
        CREATE TABLE IF NOT EXISTS public.orders (
          order_id     bigint GENERATED ALWAYS AS IDENTITY PRIMARY KEY,
          customer_id  int  NOT NULL,
          product_id   int  NOT NULL,
          product_name text,
          quantity     int  NOT NULL,
          unit_price   numeric(10,2) NOT NULL,
          total        numeric(12,2) NOT NULL,
          status       text NOT NULL DEFAULT 'confirmed',
          created_at   timestamptz NOT NULL DEFAULT now()
        );""")
    cur.execute(f'GRANT USAGE ON SCHEMA public TO "{role}";')
    for t in ("inventory", "orders"):
        cur.execute(f'GRANT SELECT, INSERT, UPDATE, DELETE ON public.{t} TO "{role}";')
    print("✅ Created public.inventory / public.orders and granted the app SP read/write")

# 3. Seed inventory from the products Delta table (idempotent)
products = (spark.table(f"{CATALOG}.{SCHEMA}.products")
                 .select("product_id", "product_name", "category", "price").collect())
with conn.cursor() as cur:
    for p in products:
        cur.execute(
            """INSERT INTO public.inventory (product_id, product_name, category, price, stock_on_hand)
               VALUES (%s, %s, %s, %s, %s)
               ON CONFLICT (product_id) DO NOTHING""",
            (p.product_id, p.product_name, p.category, float(p.price), 500),
        )
print(f"✅ Seeded inventory: {len(products)} products (stock_on_hand=500 each)")

conn.close()
print(f"\n✅ Postgres setup complete for role {role}")

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
googleapis-common-protos 1.65.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0.dev0,>=3.20.2, but you have protobuf 6.33.6 which is incompatible.
grpcio-status 1.67.0 requires protobuf<6.0dev,>=5.26.1, but you have protobuf 6.33.6 which is incompatible.


App SP role (client_id): cd61e919-c5b9-4b85-b2d1-c0887c411365
Lakebase host: ep-floral-art-d1v9vb4p.database.us-west-2.cloud.databricks.com
✅ Synced-table grants on schema lakebase_101_schema
✅ Created public.inventory / public.orders and granted the app SP read/write
✅ Seeded inventory: 12 products (stock_on_hand=500 each)

✅ Postgres setup complete for role cd61e919-c5b9-4b85-b2d1-c0887c411365
